#### 使用Pinecone：
注册账号：在Pinecone官网注册（https://www.pinecone.io）
获取API Key：在控制台获取API密钥
安装SDK：pip install pinecone
#### 核心API说明：
pinecone.create_index()：创建索引，设置维度、距离度量等
index.upsert()：插入/更新向量，支持批量操作
index.query()：向量搜索，支持元数据过滤
index.delete()：删除向量
index.describe_index_stats()：获取索引统计信息

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [ ]:
# 1. 准备数据
items = [
    {
        "type": "phone",
        "id": "用户A",
        "content": "用户A的手机号是13800001234"
    },
    {
        "type": "phone",
        "id": "用户B",
        "content": "用户B的手机号是13800005678"
    },
    {
        "type": "order",
        "id": "订单1001",
        "content": "订单1001编号203011010001"
    },
    {
        "type": "order",
        "id": "订单1002",
        "content": "订单1002编号203011010123"
    }
]

In [ ]:
# 2. Embedding
model = SentenceTransformer(r"D:\models\bge-small-zh-v1.5")

texts = [item["content"] for item in items]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
)

dimension = len(embeddings[0])

In [ ]:
# 3. 连接 Pinecone
pc = Pinecone(
    api_key = os.environ["PINECONE_API_KEY"],
)

In [ ]:
# 4. 创建 Index
index_name = "knowledge-demo"
# 这里特意没有像es删除已有 Index，除了因为 Pinecone 不支持重复建同名索引，已存在再建会报错，所以先查
# 还有就是：生产环境不能每次程序启动都把数据库删了重建。Demo 为了反复运行方便可以这么干，但真正项目里通常应该 Index 存在就直接用
# 数据更新：同一个 id → upser → 覆盖/更新
# 并且，Pinecone：没有 exists 方法，只能 list_indexes 自己判断

existing_indexes = [
    index.name                              # 返回一堆索引对象名称
    for index in pc.list_indexes()          # 列出pinecone账号下所有索引
]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    
index = pc.Index(index_name)

In [ ]:
# 5. 准备向量数据
vectors = []

for item, vector in zip(items, embeddings):
    vectors.append(
        {
            "id": item["id"],
            "values": vector.tolist(),
            "metadata": {
                "type": item["type"],
                "content": item["content"]
            }
        }
    )

In [ ]:
# 6. Upsert
index.upsert(
    vectors=vectors
)

print(f"已写入 {len(vectors)} 条向量")

In [ ]:
# 7. 查询
query_text = "用户A的电话号码是多少?"

query_vector = model.encode(
    [query_text],
    normalize_embeddings=True
)[0].tolist()


results = index.query(              # 发起搜索
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

In [ ]:
# 8. 查看结果
for rank, match in enumerate(results.matches, start=1):
    print(
        f"{rank}. "
        f"score={match.score:.4f} | "
        f"id={match.id} | "
        f"content={match.metadata.get('content')}"
    )